In [12]:
import sys
from pathlib import Path

# اضافه کردن مسیر پروژه
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

import config
from src.io import load_model, save_figure, save_table, logger
from src.models import xgb_safe_frame, xgb_feature_name_map
from src.explainability import (
    compute_feature_importance,
    get_biomarker_ranking,
    run_explainability,
)
from src.visualization import setup_style, plot_feature_importance

# 👇 ایمپورت تابع مهندسی ویژگی (بسیار مهم)
from src.feature_selection import create_engineered_features 

setup_style()

In [13]:
print("Loading model and data...")

# 1. Load the best model
model = load_model("best_model_xgboost.joblib")

# 2. Load the FINAL feature list (شامل 30 ویژگی PSO + 7 ویژگی مهندسی شده)
# ما از فایلی استفاده می‌کنیم که در انتهای نوت‌بوک 05 ذخیره شد
try:
    selected_df = pd.read_csv(config.TABLES_DIR / "selected_features_final.csv")
    selected_features = selected_df["feature"].tolist()
    print(f"Loaded {len(selected_features)} features (includes engineered ones).")
except FileNotFoundError:
    print("⚠️ 'selected_features_final.csv' not found. Falling back to base features.")
    # Fallback اگر فایل نهایی وجود نداشت
    selected_df = pd.read_csv(config.TABLES_DIR / "selected_features.csv")
    selected_features = selected_df["feature"].tolist()

# 3. Load RAW test data
X_test_raw = pd.read_csv(config.PROCESSED_DIR / "X_test_preprocessed.csv")
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

# 4. Apply Feature Engineering to Test Data (همان کاری که در Train انجام شد)
print("Applying feature engineering to Test set...")
X_test_eng, _ = create_engineered_features(X_test_raw)

# 5. Select only the features the model expects
# فیلتر کردن ستون‌ها برای اطمینان از تطابق با مدل
X_test_selected = X_test_eng[selected_features]

print(f"Model expects: {model.n_features_in_} features")
print(f"Data provided: {X_test_selected.shape[1]} features")
print(f"Test samples: {X_test_selected.shape[0]}")

2026-08-15 00:20:21 | INFO     | prostate_bcr | Loading model from D:\Prostate_BCR\core\outputs\models\best_model_xgboost.joblib


Loading model and data...
Loaded 37 features (includes engineered ones).
Applying feature engineering to Test set...


2026-08-15 00:20:22 | INFO     | prostate_bcr | Engineered: Gleason_Total, High_Risk_Gleason
2026-08-15 00:20:22 | INFO     | prostate_bcr | Engineered: Margin_x_LymphNode
2026-08-15 00:20:22 | INFO     | prostate_bcr | Engineered: T_Stage_Risk
2026-08-15 00:20:22 | INFO     | prostate_bcr | Engineered: PSA_Pathway_Score (from 7 genes)
2026-08-15 00:20:22 | INFO     | prostate_bcr | Engineered: AR_Signaling_Score (from 8 genes)
2026-08-15 00:20:22 | INFO     | prostate_bcr | Engineered: Proliferation_Score (from 7 genes)


Model expects: 37 features
Data provided: 37 features
Test samples: 86


In [14]:
# Map XGBoost-safe names back to original feature names
name_map = xgb_feature_name_map(selected_features)

importance_df = compute_feature_importance(
    model,
    selected_features,
    top_k=20,
)

print("Top 20 Features by Importance:")
print(importance_df.to_string(index=False))

2026-08-15 00:20:22 | INFO     | prostate_bcr | Extracted feature importance: 37 features


Top 20 Features by Importance:
                                             feature  importance
Primary Lymph Node Presentation Assessment Ind-3_YES    0.132058
                                            C16orf91    0.077441
                                               PRAG1    0.075200
                                               CPNE2    0.054041
                                       Gleason_Total    0.053271
                                                MND1    0.049104
                                               EPHB4    0.043216
                                        T_Stage_Risk    0.043007
                                              ERGIC1    0.041798
                                             PLEKHJ1    0.041772
                                                 SP5    0.038256
                                               HDAC4    0.034540
                                             SLCO2B1    0.029866
                                            FLJ23024    0.0

In [15]:
fig = plot_feature_importance(
    importance_df,
    top_k=20,
    title="Top 20 Features — XGBoost Importance",
    filename="feature_importance_top20.png",
)
plt.show()

2026-08-15 00:20:22 | INFO     | prostate_bcr | Saved figure → D:\Prostate_BCR\core\outputs\figures\feature_importance_top20.png
2026-08-15 00:20:22 | INFO     | prostate_bcr | Feature importance plot generated (20 features)


In [16]:
from src.preprocessing import identify_column_groups

# Identify clinical vs gene columns
clinical_cols, gene_cols = identify_column_groups(X_test_selected)

# Rank gene biomarkers
gene_importance = importance_df[importance_df["feature"].isin(gene_cols)].copy()
gene_ranking = get_biomarker_ranking(gene_importance, feature_type="gene")

# Rank clinical biomarkers
clinical_importance = importance_df[importance_df["feature"].isin(clinical_cols)].copy()
clinical_ranking = get_biomarker_ranking(clinical_importance, feature_type="clinical")

print("Top Gene Biomarkers:")
print(gene_ranking.head(10).to_string(index=False))

print("\nTop Clinical Biomarkers:")
print(clinical_ranking.head(10).to_string(index=False))

2026-08-15 00:20:22 | INFO     | prostate_bcr | Column groups: 2 clinical, 35 gene
2026-08-15 00:20:22 | INFO     | prostate_bcr | Biomarker ranking: 18 gene features
2026-08-15 00:20:22 | INFO     | prostate_bcr | Biomarker ranking: 2 clinical features


Top Gene Biomarkers:
 rank      feature  importance feature_type
    1     C16orf91    0.077441         gene
    2        PRAG1    0.075200         gene
    3        CPNE2    0.054041         gene
    4         MND1    0.049104         gene
    5        EPHB4    0.043216         gene
    6 T_Stage_Risk    0.043007         gene
    7       ERGIC1    0.041798         gene
    8      PLEKHJ1    0.041772         gene
    9          SP5    0.038256         gene
   10        HDAC4    0.034540         gene

Top Clinical Biomarkers:
 rank                                              feature  importance feature_type
    1 Primary Lymph Node Presentation Assessment Ind-3_YES    0.132058     clinical
    2                                        Gleason_Total    0.053271     clinical


In [17]:
save_table(gene_ranking, "biomarker_ranking_genes.csv", index=False)
save_table(clinical_ranking, "biomarker_ranking_clinical.csv", index=False)
save_table(importance_df, "feature_importance_full.csv", index=False)

print("Biomarker rankings saved to outputs/tables/")

2026-08-15 00:20:22 | INFO     | prostate_bcr | Saved 18 rows → D:\Prostate_BCR\core\outputs\tables\biomarker_ranking_genes.csv
2026-08-15 00:20:22 | INFO     | prostate_bcr | Saved 2 rows → D:\Prostate_BCR\core\outputs\tables\biomarker_ranking_clinical.csv
2026-08-15 00:20:22 | INFO     | prostate_bcr | Saved 20 rows → D:\Prostate_BCR\core\outputs\tables\feature_importance_full.csv


Biomarker rankings saved to outputs/tables/


In [18]:
# Run full explainability pipeline with SHAP
print("Running SHAP analysis...")
results = run_explainability(
    model=model,
    X_test=X_test_selected, 
    feature_names=selected_features,
    top_k=20,
    run_shap=True,
    sample_index=0,
)

if "shap_error" in results:
    print(f"SHAP analysis skipped: {results['shap_error']}")
else:
    print("SHAP analysis completed successfully.")

2026-08-15 00:20:22 | INFO     | prostate_bcr | Extracted feature importance: 37 features
2026-08-15 00:20:22 | INFO     | prostate_bcr | Biomarker ranking: 20 gene features
2026-08-15 00:20:23 | WARNING  | prostate_bcr | SHAP analysis failed: Duplicate column names are not supported. Duplicates found: ['High_Risk_Gleason']
2026-08-15 00:20:23 | INFO     | prostate_bcr | Explainability pipeline complete


Running SHAP analysis...
SHAP analysis skipped: Duplicate column names are not supported. Duplicates found: ['High_Risk_Gleason']


In [19]:
if "summary_plot" in results:
    fig = results["summary_plot"]
    save_figure(fig, "shap_summary_plot.png")
    plt.show()
else:
    print("SHAP summary plot not available.")

SHAP summary plot not available.


In [20]:
if "waterfall_plot" in results:
    fig = results["waterfall_plot"]
    save_figure(fig, "shap_waterfall_sample0.png")
    plt.show()
else:
    print("SHAP waterfall plot not available.")

SHAP waterfall plot not available.


In [21]:
if "shap_values" in results:
    from src.explainability import plot_shap_dependence

    top_feature = importance_df.iloc[0]["feature"]
    fig = plot_shap_dependence(
        results["shap_values"],
        X_test_selected,
        feature_name=top_feature,
        figsize=(8, 6),
    )
    save_figure(fig, f"shap_dependence_{top_feature[:20]}.png")
    plt.show()
    print(f"Dependence plot generated for: {top_feature}")
else:
    print("SHAP values not available.")

SHAP values not available.


In [22]:
# Save the final feature list (including engineered ones) just in case
pd.DataFrame({"feature": selected_features}).to_csv(
    config.TABLES_DIR / "selected_features_final.csv", index=False
)
print("Saved selected_features_final.csv")

Saved selected_features_final.csv
